In [1]:
import os
import torch
import numpy as np

SEED = 17

# modelNames = ["HGDNA_2k", "HGDNA_32k", "hyenadna-medium-160k-seqlen-hf", "DNABERT-2-117M", "nucleotide-transformer-v2-100m-multi-species", "caduceus-ps_seqlen-131k_d_model-256_n_layer-16"]
modelNames = ["HGDNA"]
dataNames = ["species_16384_cls"]
labelDict = {"human": 0, "lemur": 1, "mouse": 2, "pig": 3, "hippo": 4}

curDir = os.getcwd()

In [2]:
import matplotlib.pyplot as plt
import json
from sklearn.manifold import TSNE
from sklearn import svm
from sklearn.metrics import classification_report, matthews_corrcoef

plot_settings_dir = os.path.join(curDir, "plot_settings.json")
with open(plot_settings_dir, "r") as f:
    plot_settings = json.load(f)

plt.rcParams["font.family"] = plot_settings["font"]
plt.rcParams["font.size"] = plot_settings["fontsize"]

for dataName in dataNames:
    for modelName in modelNames:
        trainDict = torch.load(os.path.join(curDir, f"result/{modelName}/{dataName}/train.emb"), map_location="cpu", weights_only=True)
        testDict = torch.load(os.path.join(curDir, f"result/{modelName}/{dataName}/test.emb"), map_location="cpu", weights_only=True)

        trainEmb, testEmb = trainDict["pred"].to(torch.float32).numpy(), testDict["pred"].to(torch.float32).numpy()
        trainEmbTime = trainDict["wall_clock"]
        trainLabel, testLabel = trainDict["label"].numpy(), testDict["label"].numpy()

        # t-SNE plotting
        tsne = TSNE(n_components=2, random_state=SEED, n_jobs=-1)
        x_tsne = tsne.fit_transform(trainEmb)
        plt.subplots(figsize=(8, 8))
        scatter = plt.scatter(x_tsne[:, 0], x_tsne[:, 1], c=trainLabel, cmap=plt.get_cmap("tab10"), s=10)

        # remove x and y ticks
        plt.xticks([])
        plt.yticks([])
        # remove top and right spines
        plt.gca().spines["top"].set_visible(False)
        plt.gca().spines["bottom"].set_visible(False)
        plt.gca().spines["right"].set_visible(False)
        plt.gca().spines["left"].set_visible(False)

        plt.legend(handles=scatter.legend_elements()[0], labels=list(labelDict.keys()), loc="upper right")
        plt.savefig(os.path.join(curDir, f"result/{modelName}/{dataName}_emb.pdf"), dpi=600, format="pdf")
        plt.clf()
        plt.close()

        # SNV zero-shot
        model = svm.SVC(random_state=SEED)
        model.fit(trainEmb, trainLabel)
        predLabel = model.predict(testEmb)

        res = classification_report(testLabel, predLabel, target_names=list(labelDict.keys()), output_dict=True)
        mcc = matthews_corrcoef(testLabel, predLabel)

        # save to json
        res["MCC"] = mcc
        res["wall_clock"] = trainEmbTime
        with open(os.path.join(curDir, f"result/{modelName}/{dataName}_SNV.json"), "w") as f:
            f.write(json.dumps(res, indent=4))
        
        print(f"process {modelName}-{dataName} done")

process HGDNA-species_16384_cls done
